In [1]:
import os
import sys
from pathlib import Path

if 'IS_ROOT_SET' not in globals():
    project_root = Path.cwd().parent.parent.parent
    os.chdir(project_root)
    
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    
    IS_ROOT_SET = True
    print("Корневая директория инициализирована.")
else:
    print("Инициализация уже была выполнена ранее.")
    

Корневая директория инициализирована.


In [3]:
from pathlib import Path

In [8]:
from mmseg.datasets.train_dataset_for_students import TrainDatasetForStudents
from configs._base_.datasets.yandex_segementation import train_pipeline

In [9]:
from mmseg.visualization import SegLocalVisualizer
from mmseg.structures import SegDataSample
from mmengine.structures import PixelData
from mmengine.registry import init_default_scope

In [10]:
init_default_scope("mmseg")

In [ ]:
def load_practice_ds() -> TrainDatasetForStudents:

    data_root = os.path.join(project_root, "data", "train_dataset_for_students")
    data_prefix=dict(img_path=os.path.join("img", "train"), seg_map_path=os.path.join("labels", "train"))


    dataset = TrainDatasetForStudents(
        data_root=data_root, 
        data_prefix=data_prefix, 
        test_mode=False, 
        pipeline=train_pipeline,
        img_suffix=".jpg",
        seg_map_suffix=".png"
    )
    return dataset 


def plot_sample_demo(ds):    
    print(f"Загружен датасет длиной {len(ds)} элементов")
    out_dir = "viz_outputs"
    
    ds_meta = ds.metainfo


    seg_local_visualizer = SegLocalVisualizer(
        vis_backends=[dict(type='LocalVisBackend')],
        save_dir=out_dir
    )
    seg_local_visualizer.dataset_meta = dict(
        classes=ds_meta["classes"],
        palette=ds_meta["palette"]
    )

    for i in range(len(ds)):
        if i % 10 == 0:
            packed_sample = ds[i]
            img_tensor = packed_sample["inputs"]
            img = img_tensor.permute(1, 2, 0).contiguous().byte().numpy()
            
            plot_sample = packed_sample["data_samples"]

            out_file_path = os.path.join(out_dir, f"test_sample_{i}.png")

            seg_local_visualizer.add_datasample(
                name=f"test_sample_{i}",
                image=img,
                data_sample=plot_sample,
                show=False,
                draw_pred=False,
                out_file=out_file_path 
            )


ds = load_practice_ds()
plot_sample_demo(ds)

Загружен датасет длиной 200 элементов


In [ ]:
def load_practice_ds() -> PracticeDataset:
    # Инициализируем путь до корневого каталога нашего проекта 
    mmseg_root = os.path.dirname(os.path.abspath(__file__))

    # data_root поменялся и теперь указывает на наш датасет для практики  
    data_root = os.path.join(mmseg_root, "data", "practice_dataset")
    # Загружаем именно тестовый датасет, поэтому меняются префиксы  
    data_prefix=dict(img_path=os.path.join("img", "test"), seg_map_path=os.path.join("labels", "test"))
    # Пайплайн данных у нас не меняется, загружаем картинки и разметку 
    train_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations'),
    ]

    dataset = PracticeDataset(
        data_root=data_root, 
        data_prefix=data_prefix, 
        test_mode=False, 
        pipeline=train_pipeline,
        img_suffix=".jpg",  # У этого датасета картинки в формате .jpg
        seg_map_suffix=".png"
    )
    return dataset 


def plot_sample_demo(ds):    
    print(f"Загружен датасет длиной {len(ds)} элементов")

    # считываем метаинформацию  
    ds_meta = ds.metainfo


    # Подготовим визуализатор, результат будет в папке viz_outputs
    seg_local_visualizer = SegLocalVisualizer(
        vis_backends=[dict(type='LocalVisBackend')],
        save_dir="viz_outputs"
    )
    # Передаём в визуализатор метаинформацию нашего датасета 
    seg_local_visualizer.dataset_meta = dict(
        classes=ds_meta["classes"],
        palette=ds_meta["palette"]
    )

    # Оборачиваем семпл в структуру SegDataSample, совместимую с визуализатором 
    orig_sample = ds[10]
    plot_sample = SegDataSample()
    plot_sample.gt_sem_seg = PixelData(data=orig_sample["gt_seg_map"])

    # Отрисовываем семпл 
    img = orig_sample["img"]
    seg_local_visualizer.add_datasample(
        name="test_sample",
        image=img,
        data_sample=plot_sample,
        show=False,
        draw_pred=False
    )


ds = load_practice_ds()
plot_sample_demo(ds)